In [3]:
!pip install  kaggle

Defaulting to user installation because normal site-packages is not writeable


In [12]:
import os
import zipfile

# 1. Ensure Kaggle package is installed
!pip install -q kaggle

# 2. Set target path to match your project folder structure
target_dir = "/dist_home/manimaran/P6_AID"
os.makedirs(target_dir, exist_ok=True)
os.chdir(target_dir)

print("Downloading ImageNet-100 dataset from Kaggle...")
# Downloads the pre-packaged ImageNet-100 dataset directly
!kaggle datasets download -d ambityga/imagenet100

print("Unzipping dataset...")
with zipfile.ZipFile("imagenet100.zip", "r") as zip_ref:
    zip_ref.extractall("./imagenet100_raw")

# 3. Handle nesting if necessary to ensure train/ and val/ match your script's expected path
# (The dataset usually extracts to a folder like 'imagenet100' containing train.X and val.X or train/ and val/)
extracted_path = "./imagenet100_raw"
sub_items = os.listdir(extracted_path)

# Move items up if they are nested inside a subfolder
if len(sub_items) == 1 and os.path.isdir(os.path.join(extracted_path, sub_items[0])):
    inner_dir = os.path.join(extracted_path, sub_items[0])
    for item in os.listdir(inner_dir):
        os.rename(os.path.join(inner_dir, item), os.path.join(".", item))
else:
    for item in sub_items:
        os.rename(os.path.join(extracted_path, item), os.path.join(".", item))

# Clean up zip and raw temp directory
if os.path.exists("imagenet100.zip"):
    os.remove("imagenet100.zip")
if os.path.exists("imagenet100_raw"):
    os.rmdir("imagenet100_raw")

print(f"Dataset setup complete! Ready at: {os.path.abspath('./imagenet100')}")

Dataset URL: https://www.kaggle.com/datasets/ambityga/imagenet100
License(s): unknown
Resuming from 629145600 bytes (16664792062 bytes left)...
 64%|████████████████████████▎             | 10.3G/16.1G [53:01<31:28, 3.29MB/s]

Connection error: ChunkedEncodingError: ('Connection broken: IncompleteRead(10459397122 bytes read, 6205394940 more expected)', IncompleteRead(10459397122 bytes read, 6205394940 more expected))
Retrying in 2.4 seconds... (attempt 1/5)
Retry 1/5: Resuming from 11087642624 bytes (6206295038 bytes left)...
100%|██████████████████████████████████████| 16.1G/16.1G [27:38<00:00, 3.74MB/s]

Unzipping dataset...
Dataset setup complete! Ready at: /dist_home/manimaran/P6_AID/imagenet100


In [13]:
import os

root = "./imagenet100"
print("Train path exists:", os.path.exists(os.path.join(root, "train")))
if os.path.exists(os.path.join(root, "train")):
    subfolders = os.listdir(os.path.join(root, "train"))
    print("Number of train class folders:", len(subfolders))
    if len(subfolders) > 0:
        sample_folder = os.path.join(root, "train", subfolders[0])
        print(f"Images in first folder ({subfolders[0]}):", len(os.listdir(sample_folder)))

Train path exists: False


In [17]:
!python train_baseline_in100.py --model deit_tiny



[ImageNet-100] 100 classes | train 125000 | val(held-out train) 5000 | test(official val) 5000
/dist_home/common-apps/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
[head] WARNING: random head (needs the full 1000-class train folder + pretrained weights to reuse the classifier rows); train longer
[start] val acc before fine-tuning: 0.78%
ep 1/5 loss 1.626 val acc 86.92%
ep 2/5 loss 1.344 val acc 87.68%
ep 3/5 loss 1.297 val acc 88.20%
ep 4/5 loss 1.264 val acc 88.82%
ep 5/5 loss 1.242 val acc 88.98%
best val 88.98% | TEST acc 86.14% | saved ./phase1_in100_outputs/deit_ti

In [19]:
!python train_baseline_in100.py --model mobilevit_xs 

[ImageNet-100] 100 classes | train 125000 | val(held-out train) 5000 | test(official val) 5000
/dist_home/common-apps/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
[head] WARNING: random head (needs the full 1000-class train folder + pretrained weights to reuse the classifier rows); train longer
[start] val acc before fine-tuning: 1.24%
ep 1/5 loss 2.203 val acc 85.94%
ep 2/5 loss 1.576 val acc 87.40%
ep 3/5 loss 1.505 val acc 88.40%
ep 4/5 loss 1.473 val acc 88.60%
ep 5/5 loss 1.456 val acc 88.62%
best val 88.62% | TEST acc 84.48% | saved ./phase1_in100_outputs/mobilev

In [39]:
!python run_ee.py --model deit_tiny --dataset imagenet100 --reuse --val_per_class 0 --workers 5 --lambdas 0.15 --seeds 42 --reuse  --onnx --onnx_samples 5000

[ImageNet-100] 100 classes | train 130000 | val(held-out train) 0 | test(official val) 5000
[deit_tiny] baseline val 86.72%  test 85.60%  (no flip-TTA)
-> 1 training run(s): lambdas [0.15] x seeds [42], 15 epochs each

##### lambda_hw=0.15 seed 42: reusing ./ee_outputs/imagenet100/deit_tiny/lhw0.15/seed42/deit_tiny_lhw0.15_s42_best.pth

=== deit_tiny_lhw0.15_s42 | costs: cpu [0.338, 0.669, 1.0] | baseline test acc 85.60% (val 86.72%) | head test acc [68.12, 81.64, 86.32]
router       budget test acc   drop  relLat  saving%  oracle%  exits%
---------------------------------------------------------------------
msp            0.25    85.80  -0.20   0.673     32.7     67.1   31.3  36.1  32.6
msp            0.50    84.76   0.84   0.636     36.4     74.7   30.0  50.1  19.9
msp            1.00    84.36   1.24   0.595     40.5     83.0   41.6  39.1  19.3
msp            2.00    83.64   1.96   0.563     43.7     89.7   46.7  38.7  14.6
margin         0.25    85.76  -0.16   0.675     32.5     66.

In [40]:
!python run_extra.py --model deit_tiny --dataset imagenet100  --val_per_class 0 --workers 5 --lambdas 0.15 --seeds 42 43 --bootstrap 500

[ImageNet-100] 100 classes | train 130000 | val(held-out train) 0 | test(official val) 5000
[deit_tiny lambda 0.15 seed 42] evaluating ...
[deit_tiny lambda 0.15 seed 43] evaluating ...
[bootstrap] 500 resamples x 2 seed(s) ...

deit_tiny | lambda_hw 0.15 | 2 seed(s) | savings vs each model's OWN final head | thresholds on val-half-B

=== latency saving % at test accuracy within 0.5 pt of baseline (matched accuracy), mean ± std over seeds ===
router              saving %           vs margin    wins
msp               nan ± nan            +nan ± nan     0/0  <-- only 0 of 2 seeds reached this accuracy
margin            nan ± nan           (reference)       -  <-- only 0 of 2 seeds reached this accuracy
entropy           nan ± nan            +nan ± nan     0/0  <-- only 0 of 2 seeds reached this accuracy
conf_gate         nan ± nan            +nan ± nan     0/0  <-- only 0 of 2 seeds reached this accuracy
vib_latent       13.3 ± 0.5            +nan ± nan     0/2
vib_gate          nan ± na

In [37]:
!python run_ee.py --model mobilevit_xs --dataset imagenet100 --reuse --val_per_class 0 --workers 5 --lambdas 0.15 --seeds 42 43 --onnx --onnx_samples 5000

[ImageNet-100] 100 classes | train 130000 | val(held-out train) 0 | test(official val) 5000
[mobilevit_xs] baseline val 85.04%  test 83.96%  (no flip-TTA)
-> 2 training run(s): lambdas [0.15] x seeds [42, 43], 15 epochs each

##### lambda_hw=0.15 seed 42: training 15 epochs
[mobilevit_xs_lhw0.15_s42] baseline val acc 85.04% | exits [2, 3] | stage costs used [0.708, 0.943, 1.0] | measured { cpu: [0.708, 0.943, 1.0], gpu: [0.381, 0.774, 1.0] }
[mobilevit_xs_lhw0.15_s42] ep 01/15 loss 3.027 | head acc 47.6/72.4/84.7 | acc/relLat per router: msp 84.6/0.969 | margin 84.6/0.970 | entropy 84.6/0.985 | conf_gate 84.6/0.970 | vib_latent 84.6/0.999 | vib_gate 84.6/0.974
[mobilevit_xs_lhw0.15_s42] ep 02/15 loss 2.782 | head acc 55.6/76.4/84.6 | acc/relLat per router: msp 84.6/0.970 | margin 84.6/0.956 | entropy 84.6/0.970 | conf_gate 84.6/0.956 | vib_latent 84.6/0.999 | vib_gate 84.6/0.964
[mobilevit_xs_lhw0.15_s42] ep 03/15 loss 2.725 | head acc 58.5/77.8/84.7 | acc/relLat per router: msp 84.6/0

In [38]:
!python run_extra.py --model mobilevit_xs --dataset imagenet100  --val_per_class 0 --workers 5 --lambdas 0.15 --seeds 42 43 --bootstrap 500

[ImageNet-100] 100 classes | train 130000 | val(held-out train) 0 | test(official val) 5000
[mobilevit_xs lambda 0.15 seed 42] evaluating ...
[mobilevit_xs lambda 0.15 seed 43] evaluating ...
[bootstrap] 500 resamples x 2 seed(s) ...

mobilevit_xs | lambda_hw 0.15 | 2 seed(s) | savings vs each model's OWN final head | thresholds on val-half-B

=== latency saving % at test accuracy within 0.5 pt of baseline (matched accuracy), mean ± std over seeds ===
router              saving %           vs margin    wins
msp              11.2 ± 1.4            -1.2 ± 0.0     0/2
margin           12.5 ± 1.4           (reference)       -
entropy           7.9 ± 1.0            -4.5 ± 0.4     0/2
conf_gate        12.7 ± 1.5            +0.2 ± 0.1     2/2
vib_latent        2.9 ± 0.1            -9.5 ± 1.5     0/2
vib_gate         11.7 ± 1.9            -0.8 ± 0.4     0/2
lr_conf          12.2 ± 1.2            -0.3 ± 0.2     0/2
lr_traj          12.2 ± 1.2            -0.3 ± 0.2     0/2
lr_agree         12.4 ±